# Rain Prediction — Logistic Regression

End-to-end logistic regression pipeline for predicting next-day rainfall (`RainTomorrow`) on the MICE-preprocessed WeatherAUS dataset.

Each section explains *what* the step does. Numerical outputs are mirrored to `LR_result.txt`; every figure is saved as a PNG under `figures/`. Both artifacts feed a written report after the notebook is executed.


## 1. Imports and constants


In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV, TimeSeriesSplit, learning_curve,
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
    roc_curve, roc_auc_score, precision_recall_curve, auc,
    brier_score_loss, log_loss,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

RANDOM_STATE = 42
TEST_CUTOFF = pd.Timestamp('2016-01-01')   # train < cutoff <= test


### Output setup

A `figures/` directory and an `LR_result.txt` file are created next to this notebook. `sys.stdout` is teed so every `print(...)` is mirrored to the results file. Each plot is saved with `save_fig(...)` before `plt.show()`.


In [ ]:
# Create output locations
os.makedirs('figures', exist_ok=True)

_LR_RESULT_PATH = 'LR_result.txt'
_LR_RESULT_FILE = open(_LR_RESULT_PATH, 'w', encoding='utf-8')

class _Tee:
    """Write to both real stdout and the results file."""
    def __init__(self, *streams):
        self._streams = streams
    def write(self, data):
        for s in self._streams:
            s.write(data)
    def flush(self):
        for s in self._streams:
            s.flush()
    def isatty(self):
        return False

_ORIG_STDOUT = sys.stdout
sys.stdout = _Tee(_ORIG_STDOUT, _LR_RESULT_FILE)

print(f'# Logistic Regression — numerical outputs')
print(f'# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

def log_section(title):
    """Print a section banner so LR_result.txt is easy to skim."""
    bar = '=' * 72
    print(f'\n{bar}\n{title}\n{bar}')

def save_fig(name, fig=None):
    """Save the current (or given) figure to figures/<name>.png at 150 dpi."""
    if fig is None:
        fig = plt.gcf()
    path = f'figures/{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  [saved {path}]')


## 2. Data loading

Input is the post-MICE dataset produced by the preprocessing notebook: 138,820 rows × 32 columns, no nulls. The `Date` column is the splitting axis for §4 and is parsed first.


In [ ]:
log_section('2. Data loading')

DATA_PATH = 'data/WeatherAUS_preprocessed_MICEcyclic.csv'
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
df['Date'] = pd.to_datetime(df['Date'])
print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Locations : {df["Location"].nunique()}')
df['RainTomorrow'] = df['RainTomorrow'].astype(bool)


## 3. Feature engineering

Three categories of columns are dropped before training:

- **Redundant pair members.** `Humidity9am` is dropped in favour of the more-predictive `Humidity3pm`.
- **Geographic identity.** `Location` is removed so the model relies on meteorological signal alone, not per-station base rates.
- **Discrete wind-direction labels.** The cyclic sine/cosine pairs created during preprocessing already carry the same information without injecting a 16-level categorical.

The cyclic date features (`Date_sin`, `Date_cos`) should already exist in the input CSV from preprocessing; we recompute them here only to make the notebook runnable on inputs that lack them.


In [ ]:
if 'Date_sin' not in df.columns:
    day_of_year = df['Date'].dt.dayofyear
    df['Date_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    df['Date_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)


In [ ]:
log_section('3. Feature engineering')

DROP_COLS = [
    'Date',
    'Location',                              # no per-station dummies
    'WindGustDir', 'WindDir9am', 'WindDir3pm',  # discrete labels; sin/cos retained
    'RainToday',                             # near-leakage; the model receives the rest of today
    'RainTomorrow',                          # target
    'Humidity9am',                           # keep Humidity3pm (the stronger predictor)
    'Month', 'weather_trend_group', 'Sunshine_trend_group',
]

X_full = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()
y_full = df['RainTomorrow']

# Coerce any residual non-numeric columns; we keep only numeric features.
X_full = X_full.select_dtypes(include=[np.number])

print(f'Feature matrix: {X_full.shape}')
print(f'Columns ({len(X_full.columns)}):')
for c in X_full.columns:
    print(f'  - {c}')
print(f'\nClass distribution (RainTomorrow):')
print(y_full.value_counts().to_string())
print(f'Rain proportion: {y_full.mean():.4f}')


## 4. Chronological train/test split

The EDA flagged within-station temporal autocorrelation as the IID violation in this dataset. A random k-fold or stratified split places day $t$ from station $s$ in training and day $t+1$ from the same station in test; the lag-1 autocorrelation $r \approx 0.46$ established in EDA §6 then becomes a free signal that does not correspond to anything achievable in deployment.

We split chronologically: train < 2016-01-01, test from 2016-01-01 onward. The cutoff places approximately 85% of the data in train (2007-11 to 2015-12, ~8 years) and 15% in test (2016-01 to 2017-06, ~1.5 years). All in-training CV uses `TimeSeriesSplit`, which produces ascending-time folds — every validation fold contains only dates later than its training fold. This is the standard one-step-ahead forecasting evaluation for stationary time series.


In [ ]:
log_section('4. Chronological train/test split')

df_sorted = df.sort_values('Date').reset_index(drop=True)
X_sorted = X_full.loc[df_sorted.index].reset_index(drop=True)
y_sorted = y_full.loc[df_sorted.index].reset_index(drop=True)
dates_sorted = df_sorted['Date']

train_mask = dates_sorted < TEST_CUTOFF
X_train = X_sorted[train_mask].reset_index(drop=True)
X_test  = X_sorted[~train_mask].reset_index(drop=True)
y_train = y_sorted[train_mask].reset_index(drop=True)
y_test  = y_sorted[~train_mask].reset_index(drop=True)

print(f'Train: {X_train.shape[0]:,} rows | dates {dates_sorted[train_mask].min().date()} to {dates_sorted[train_mask].max().date()}')
print(f'Test : {X_test.shape[0]:,} rows | dates {dates_sorted[~train_mask].min().date()} to {dates_sorted[~train_mask].max().date()}')
print(f'Train rain proportion: {y_train.mean():.4%}')
print(f'Test  rain proportion: {y_test.mean():.4%}')
print(f'Drift (test - train) : {(y_test.mean() - y_train.mean()) * 100:+.2f} percentage points')


In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)
print(f'Scaled {X_train_scaled.shape[1]} numeric features (fit on train only).')


## 5. Baseline model

Default `LogisticRegression` with no class weighting and the default 0.5 threshold. This establishes the floor: any subsequent improvement must demonstrate a measurable gain over this configuration to justify its added complexity.


In [ ]:
log_section('5. Baseline model')

baseline = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train_scaled, y_train)
y_pred_baseline = baseline.predict(X_test_scaled)
y_proba_baseline = baseline.predict_proba(X_test_scaled)[:, 1]

print('Baseline classification report:')
print(classification_report(y_test, y_pred_baseline))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_baseline):.4f}')


In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
cm = confusion_matrix(y_test, y_pred_baseline)
ConfusionMatrixDisplay(cm, display_labels=baseline.classes_).plot(cmap='Blues', ax=ax)
ax.set_title('Baseline — confusion matrix (chronological test)')
plt.tight_layout()
save_fig('01_baseline_confusion_matrix', fig)
plt.show()


## 6. Hyperparameter tuning with TimeSeriesSplit

`GridSearchCV` over $C$, penalty, and `class_weight`, scored on F1 of the rain class. The CV iterator is `TimeSeriesSplit(n_splits=5)` rather than the default stratified k-fold — every validation fold contains only training-period dates later than its training fold, mirroring the deployment task of forecasting tomorrow's rain from today's observations.


In [ ]:
log_section('6. Hyperparameter tuning')

param_grid = {
    'C':            [0.01, 0.1, 1, 10],
    'penalty':      ['l1', 'l2'],
    'solver':       ['liblinear'],
    'class_weight': [None, 'balanced'],
}

ts_cv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid,
    cv=ts_cv,
    scoring='f1',
    n_jobs=-1,
)
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
print(f'Best params : {grid_search.best_params_}')
print(f'Best CV F1  : {grid_search.best_score_:.4f}')


In [ ]:
y_pred_tuned = best_model.predict(X_test_scaled)
y_proba_tuned = best_model.predict_proba(X_test_scaled)[:, 1]

print('Tuned model — classification report:')
print(classification_report(y_test, y_pred_tuned))
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba_tuned):.4f}')

# Diagnostic on coefficient sparsity (only meaningful under L1).
if grid_search.best_params_['penalty'] == 'l1':
    n_zero = int(np.sum(best_model.coef_ == 0))
    n_total = best_model.coef_.shape[1]
    print(f'L1 zeroed {n_zero}/{n_total} features at C={grid_search.best_params_["C"]}.')
else:
    print(f'L2 selected; coefficient sparsity is not applicable (L2 produces no exact zeros).')


## 7. Class imbalance — three strategies at fixed $C = 1, \ell_1$

Three orthogonal handlings of the ~78/22 imbalance: loss reweighting (`class_weight='balanced'`), threshold tuning (selected separately in §8), and SMOTE oversampling. All three are compared at the same regularization configuration so the numbers are directly comparable.

SMOTE under chronological evaluation is debatable — synthetic minority points are interpolations between training-period rainy days, and applying them to a test period that contains genuinely new weather can degrade rather than improve generalization. We include it for completeness and report the trade-off.

Per-strategy confusion matrices follow the comparison table.


In [ ]:
log_section('7. Class imbalance — three strategies')

results = []

# 7a — default class weighting
m1 = LogisticRegression(C=1, penalty='l1', solver='liblinear',
                        max_iter=1000, random_state=RANDOM_STATE)
m1.fit(X_train_scaled, y_train)
p1 = m1.predict(X_test_scaled)
results.append({'config': 'default', 'F1': f1_score(y_test, p1),
                'precision': precision_score(y_test, p1),
                'recall': recall_score(y_test, p1)})

# 7b — balanced class weights
m2 = LogisticRegression(C=1, penalty='l1', solver='liblinear',
                        class_weight='balanced',
                        max_iter=1000, random_state=RANDOM_STATE)
m2.fit(X_train_scaled, y_train)
p2 = m2.predict(X_test_scaled)
results.append({'config': 'balanced weights', 'F1': f1_score(y_test, p2),
                'precision': precision_score(y_test, p2),
                'recall': recall_score(y_test, p2)})

# 7c — SMOTE
m3 = None
p3 = None
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=RANDOM_STATE)
    X_sm, y_sm = smote.fit_resample(X_train_scaled, y_train)
    m3 = LogisticRegression(C=1, penalty='l1', solver='liblinear',
                            max_iter=1000, random_state=RANDOM_STATE)
    m3.fit(X_sm, y_sm)
    p3 = m3.predict(X_test_scaled)
    results.append({'config': 'SMOTE', 'F1': f1_score(y_test, p3),
                    'precision': precision_score(y_test, p3),
                    'recall': recall_score(y_test, p3)})
    print(f'SMOTE training set: {len(y_sm):,} rows ({pd.Series(y_sm).mean():.0%} rain)')
except ImportError:
    print('imblearn not installed; SMOTE skipped.')

imbalance_df = pd.DataFrame(results).round(3)
print('\nImbalance comparison:')
print(imbalance_df.to_string(index=False))


In [ ]:
# Confusion matrix — balanced class weights
fig, ax = plt.subplots(figsize=(5.5, 4.5))
cm = confusion_matrix(y_test, p2)
ConfusionMatrixDisplay(cm, display_labels=m2.classes_).plot(cmap='Blues', ax=ax)
ax.set_title('Balanced weights (C=1, L1) — confusion matrix')
plt.tight_layout()
save_fig('02_balanced_weights_confusion_matrix', fig)
plt.show()


In [ ]:
# Confusion matrix — SMOTE
if m3 is not None and p3 is not None:
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    cm = confusion_matrix(y_test, p3)
    ConfusionMatrixDisplay(cm, display_labels=m3.classes_).plot(cmap='Blues', ax=ax)
    ax.set_title('SMOTE (C=1, L1) — confusion matrix')
    plt.tight_layout()
    save_fig('03_smote_confusion_matrix', fig)
    plt.show()
else:
    print('SMOTE confusion matrix skipped (imblearn unavailable).')


## 8. Threshold selection on training folds (TimeSeriesSplit)

The decision threshold is selected by 5-fold `TimeSeriesSplit` CV on the training set. The selection criterion is *not* maximum F1; it is the threshold that minimizes the precision–recall imbalance, defined as
$$D(t) = \big| P(t) - F_1(t) \big| + \big| F_1(t) - R(t) \big|,$$
where $P$, $R$, $F_1$ are the mean validation precision, recall, and F1 across the five folds at threshold $t$. Because $F_1 = 2PR/(P+R)$ is the harmonic mean of $P$ and $R$, $D(t) = 0$ if and only if $P = R$, and $D(t) > 0$ otherwise — so $\arg\min_t D(t)$ is the threshold at which precision and recall cross.

This differs from $\arg\max_t F_1(t)$ in a substantive way: F1-max picks the threshold where the marginal gain in $P$ matches the marginal loss in $R$ (or vice versa), which on imbalanced data with a recall-heavy model typically lands at a low threshold and a recall-skewed operating point. The PR-crossover criterion picks the threshold where the two are equal, which gives a single number ($P = R$) describing the operating point and is the relevant criterion when the deployment cost of a false negative equals that of a false positive.

The threshold sweep is restricted to $[0.60, 0.80]$ at step $0.01$. The lower bound is set above the conventional 0.5 because the tuned model is trained under `class_weight='balanced'`: its probability scale is shifted upward (mean predicted $\approx 0.35$ against an empirical base rate of $0.20$, as confirmed in §11), so the F1-relevant region of the threshold axis lies above 0.5 rather than below it.


In [ ]:
log_section('8. Threshold selection (TimeSeriesSplit CV)')

thresholds = np.round(np.arange(0.6, 0.8, 0.01), 2)

fold_records = []
for tr_idx, val_idx in ts_cv.split(X_train_scaled):
    fm = LogisticRegression(**best_model.get_params())
    fm.fit(X_train_scaled.iloc[tr_idx], y_train.iloc[tr_idx])
    fold_records.append((
        y_train.iloc[val_idx].values,
        fm.predict_proba(X_train_scaled.iloc[val_idx])[:, 1],
    ))

rows = []
for t in thresholds:
    f1s = [f1_score(yt, yp >= t) for yt, yp in fold_records]
    precs = [precision_score(yt, yp >= t, zero_division=0) for yt, yp in fold_records]
    recs = [recall_score(yt, yp >= t) for yt, yp in fold_records]
    rows.append({'threshold': t, 'CV_F1_mean': np.mean(f1s), 'CV_F1_std': np.std(f1s),
                 'CV_precision': np.mean(precs), 'CV_recall': np.mean(recs)})
thresh_df = pd.DataFrame(rows)


In [ ]:
thresh_df['diff'] = abs(thresh_df['CV_precision'] - thresh_df['CV_F1_mean']) + abs(thresh_df['CV_F1_mean'] - thresh_df['CV_recall'])


In [ ]:
best_threshold = float(thresh_df.loc[thresh_df['diff'].idxmin(), 'threshold'])
print(f'CV-selected threshold: {best_threshold}')
print('\nThreshold sweep table:')
print(thresh_df.to_string(index=False, float_format='%.3f'))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresh_df['threshold'], thresh_df['CV_precision'], 'o-', label='CV precision')
ax.plot(thresh_df['threshold'], thresh_df['CV_recall'], 'o-', label='CV recall')
ax.plot(thresh_df['threshold'], thresh_df['CV_F1_mean'], 's--', label='CV F1')
ax.fill_between(thresh_df['threshold'],
                thresh_df['CV_F1_mean'] - thresh_df['CV_F1_std'],
                thresh_df['CV_F1_mean'] + thresh_df['CV_F1_std'],
                alpha=0.15, label='F1 +/- 1 SD')
ax.axvline(best_threshold, color='red', linestyle=':', label=f'Selected = {best_threshold}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold sweep — TimeSeriesSplit CV on training set')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
save_fig('04_threshold_sweep', fig)
plt.show()

# Single-touch test evaluation at the selected threshold
y_pred_thr = (y_proba_tuned >= best_threshold)
print(f'\nTest evaluation at threshold {best_threshold}:')
print(classification_report(y_test, y_pred_thr))


## 9. Interaction terms

Top-8 numeric features by absolute coefficient are selected from the tuned model and passed through `PolynomialFeatures(degree=2, interaction_only=True)` to generate $\binom{8}{2} = 28$ pairwise cross-terms. The diagnostic message reports the zeroed-feature count under the actual penalty in use ($\ell_1$ produces exact zeros; $\ell_2$ does not).


In [ ]:
log_section('9. Interaction terms')

k = 8
importance_df = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'AbsCoef': np.abs(best_model.coef_[0]),
}).sort_values('AbsCoef', ascending=False)
top_k = importance_df['Feature'].head(k).tolist()
print(f'Top-{k} numeric features: {top_k}')

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly.fit(X_train_scaled[top_k])
all_names = list(poly.get_feature_names_out(top_k))
cross_names = [n for n in all_names if ' ' in n]
cross_idx = [all_names.index(n) for n in cross_names]

def append_cross(X_base):
    arr = poly.transform(X_base[top_k])
    return pd.concat([X_base,
                      pd.DataFrame(arr[:, cross_idx], columns=cross_names, index=X_base.index)],
                     axis=1)

X_train_int = append_cross(X_train_scaled)
X_test_int = append_cross(X_test_scaled)

log_reg_int = LogisticRegression(**best_model.get_params())
log_reg_int.fit(X_train_int, y_train)
y_pred_int = log_reg_int.predict(X_test_int)
y_proba_int = log_reg_int.predict_proba(X_test_int)[:, 1]

# Diagnostic — penalty-aware
penalty_used = log_reg_int.get_params()['penalty']
coef_cross = log_reg_int.coef_[0][len(X_train_scaled.columns):]
n_zero_cross = int(np.sum(coef_cross == 0))
if penalty_used == 'l1':
    print(f'Penalty = L1: zeroed {n_zero_cross}/{len(cross_names)} interaction terms.')
else:
    print(f'Penalty = L2: zeroed-coefficient count is structurally 0 under L2 '
          f'(L2 shrinks toward zero but does not produce exact zeros). '
          f'All {len(cross_names)} interaction terms are retained.')

print(f'\nWith {len(cross_names)} interaction terms — classification report:')
print(classification_report(y_test, y_pred_int))
print(f'F1 (rain): {f1_score(y_test, y_pred_int):.4f}')
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba_int):.4f}')


## 10. Bootstrap coefficient stability

Two views of the tuned model's coefficients:

- **Top-20 absolute coefficients of the single fitted model** — a quick ranking of which features the optimizer relied on most.
- **$N = 200$ bootstrap resamples** on the training set, refitting the tuned model on each resample and storing both the coefficient vector and the predicted test-set probabilities. Bootstrap quantiles give 95% CIs; bagged probabilities give an ensemble prediction at the §8 threshold.

Standard bootstrap is technically agnostic to the chronological ordering — sampling with replacement scrambles dates — but for coefficient stability under the stationary-physics assumption this is acceptable. A block bootstrap would be needed if the autocorrelation structure itself were to be characterized.


In [ ]:
log_section('10. Bootstrap coefficient stability')

# Top-20 features by absolute coefficient of the single best fitted model
coef_importance_full = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': best_model.coef_[0],
    'AbsCoefficient': np.abs(best_model.coef_[0]),
}).sort_values('AbsCoefficient', ascending=False)

print('Top 20 features by absolute coefficient (best_model):')
print(coef_importance_full.head(20).to_string(index=False, float_format='%.4f'))

fig = plt.figure(figsize=(9, 7))
sns.barplot(
    data=coef_importance_full.head(20),
    x='AbsCoefficient', y='Feature',
    hue='Feature', palette='magma', legend=False,
)
plt.title('Top 20 features by absolute coefficient (best_model)')
plt.xlabel('|Coefficient|')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
save_fig('05_top20_coefficients', fig)
plt.show()


In [ ]:
N_BOOTSTRAP = 200
boot_coefs = []
boot_test_probs = []

for i in range(N_BOOTSTRAP):
    X_bs, y_bs = resample(X_train_scaled, y_train, replace=True, random_state=i)
    m = LogisticRegression(**best_model.get_params())
    m.fit(X_bs, y_bs)
    boot_coefs.append(m.coef_[0])
    boot_test_probs.append(m.predict_proba(X_test_scaled)[:, 1])

boot_df = pd.DataFrame(boot_coefs, columns=X_train_scaled.columns)
boot_summary = pd.DataFrame({
    'mean':  boot_df.mean(),
    'std':   boot_df.std(),
    'q025':  boot_df.quantile(0.025),
    'q975':  boot_df.quantile(0.975),
    'abs_mean': boot_df.mean().abs(),
}).sort_values('abs_mean', ascending=False)

print(f'\nBootstrap complete (N={N_BOOTSTRAP}). Top-15 features:')
print(boot_summary.head(15).round(4).to_string())


In [ ]:
top_n = 12
top_feats = boot_summary.head(top_n).index

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.boxplot(data=boot_df[top_feats], orient='h', ax=axes[0], palette='vlag')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title(f'Top-{top_n} bootstrap coefficient distributions')
axes[0].set_xlabel('Coefficient value')

ci_data = boot_summary.loc[top_feats]
axes[1].errorbar(
    x=ci_data['mean'], y=ci_data.index,
    xerr=[ci_data['mean'] - ci_data['q025'], ci_data['q975'] - ci_data['mean']],
    fmt='o', ecolor='gray', capsize=4,
)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('95% bootstrap CIs')
axes[1].set_xlabel('Coefficient value')
axes[1].invert_yaxis()
plt.tight_layout()
save_fig('06_bootstrap_stability', fig)
plt.show()


In [ ]:
y_proba_bagged = np.mean(boot_test_probs, axis=0)
y_pred_bagged = (y_proba_bagged >= best_threshold)

print(f'\nBagged bootstrap (N={N_BOOTSTRAP}, threshold={best_threshold}) — classification report:')
print(classification_report(y_test, y_pred_bagged))
print(f'ROC-AUC : {roc_auc_score(y_test, y_proba_bagged):.4f}')


## 11. Calibration analysis

ROC-AUC and PR-AUC measure *discrimination* — the model's ability to rank rainy days above dry days — and are invariant to monotone transforms of the predicted probability. They therefore say nothing about whether $\hat{p} = 0.7$ corresponds to a 70% empirical positive rate. A miscalibrated model can rank perfectly while producing systematically biased probabilities, which becomes a problem when the probability itself is used downstream (decision thresholds tied to expected costs, ensemble averaging, probabilistic forecasts).

Three diagnostics:

- **Reliability diagram**: bin predicted probabilities, plot mean predicted vs empirical positive rate per bin. Perfect calibration is the diagonal $y = x$.
- **Brier score**: $\frac{1}{N} \sum_i (\hat{p}_i - y_i)^2$. A strictly proper scoring rule that decomposes into reliability + resolution + uncertainty.
- **Expected calibration error**: $\text{ECE} = \sum_b (|B_b|/N) \, |\text{acc}(B_b) - \text{conf}(B_b)|$, the weighted gap between empirical accuracy and mean confidence across $b$ bins.

An L2-regularized logistic model trained without class reweighting is typically well-calibrated because the cross-entropy loss is itself the calibration loss. Under `class_weight='balanced'`, however, the loss is no longer the maximum-likelihood objective: the reweighting biases probabilities away from the empirical base rate toward the balanced rate, producing a known overestimation of the minority-class probability. We measure this directly across the baseline, tuned, and bagged-bootstrap probability sources.


In [ ]:
log_section('11. Calibration analysis')

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.digitize(y_prob, bin_edges[1:-1])
    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.sum() > 0:
            acc = y_true[mask].mean()
            conf = y_prob[mask].mean()
            ece += (mask.sum() / n) * abs(acc - conf)
    return ece

# Compute calibration on three probability sources
probs = {
    'baseline (no reweight)': y_proba_baseline,
    'tuned (best_model)':     y_proba_tuned,
    'bagged bootstrap':       y_proba_bagged,
}

calib_metrics = []
for name, p in probs.items():
    calib_metrics.append({
        'model':  name,
        'Brier':  brier_score_loss(y_test, p),
        'logloss': log_loss(y_test, np.clip(p, 1e-9, 1 - 1e-9)),
        'ECE_10': expected_calibration_error(y_test.values, p, n_bins=10),
        'mean_pred': float(np.mean(p)),
        'empirical_rate': float(y_test.mean()),
    })
calib_table = pd.DataFrame(calib_metrics).round(4)
print('\nCalibration metrics (three probability sources):')
print(calib_table.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.plot([0, 1], [0, 1], 'k:', label='Perfectly calibrated', alpha=0.7)
for name, p in probs.items():
    prob_true, prob_pred = calibration_curve(y_test, p, n_bins=10, strategy='quantile')
    ax.plot(prob_pred, prob_true, 'o-', label=name, alpha=0.85)
ax.set_xlabel('Mean predicted probability (per bin)')
ax.set_ylabel('Empirical positive rate (per bin)')
ax.set_title('Reliability diagram — three probability sources')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig('07_reliability_diagram', fig)
plt.show()


In [ ]:
# Post-hoc recalibration: split the training period chronologically into
# (refit, calibrate), fit isotonic and sigmoid mappings on the calibration slice,
# evaluate on the test set.

n_train = len(X_train_scaled)
calib_split = int(n_train * 0.85)
X_refit = X_train_scaled.iloc[:calib_split]
y_refit = y_train.iloc[:calib_split]
X_calib = X_train_scaled.iloc[calib_split:]
y_calib = y_train.iloc[calib_split:]

refit_model = LogisticRegression(**best_model.get_params())
refit_model.fit(X_refit, y_refit)

calib_iso = CalibratedClassifierCV(refit_model, method='isotonic', cv='prefit')
calib_iso.fit(X_calib, y_calib)

calib_sig = CalibratedClassifierCV(refit_model, method='sigmoid', cv='prefit')
calib_sig.fit(X_calib, y_calib)

y_proba_iso = calib_iso.predict_proba(X_test_scaled)[:, 1]
y_proba_sig = calib_sig.predict_proba(X_test_scaled)[:, 1]

rows = []
for name, p in [('refit (uncalibrated)', refit_model.predict_proba(X_test_scaled)[:, 1]),
                ('isotonic recalibrated', y_proba_iso),
                ('sigmoid (Platt) recalibrated', y_proba_sig)]:
    rows.append({
        'method': name,
        'Brier':  brier_score_loss(y_test, p),
        'logloss': log_loss(y_test, np.clip(p, 1e-9, 1 - 1e-9)),
        'ECE_10': expected_calibration_error(y_test.values, p, n_bins=10),
        'mean_pred': float(np.mean(p)),
        'ROC_AUC': roc_auc_score(y_test, p),
    })
recalib_table = pd.DataFrame(rows).round(4)
print('\nPost-hoc recalibration metrics:')
print(recalib_table.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.plot([0, 1], [0, 1], 'k:', alpha=0.7, label='Perfect')
for name, p in [('uncalibrated', refit_model.predict_proba(X_test_scaled)[:, 1]),
                ('isotonic', y_proba_iso),
                ('Platt sigmoid', y_proba_sig)]:
    prob_true, prob_pred = calibration_curve(y_test, p, n_bins=10, strategy='quantile')
    ax.plot(prob_pred, prob_true, 'o-', label=name, alpha=0.85)
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Empirical positive rate')
ax.set_title('Reliability diagram — post-hoc recalibration')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)
plt.tight_layout()
save_fig('08_recalibration_reliability_diagram', fig)
plt.show()


## 12. Final evaluation curves

ROC and Precision–Recall curves for the three principal probability sources (tuned, bagged bootstrap, isotonic-recalibrated), followed by a learning curve under TimeSeriesSplit CV that shows how training and validation F1 evolve with sample size.


In [ ]:
log_section('12. Final evaluation curves')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC
roc_aucs = {}
for name, p in [('tuned', y_proba_tuned),
                ('bagged', y_proba_bagged),
                ('isotonic recalibrated', y_proba_iso)]:
    fpr, tpr, _ = roc_curve(y_test, p)
    auc_v = roc_auc_score(y_test, p)
    roc_aucs[name] = auc_v
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_v:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC — chronological test set')
axes[0].legend(loc='lower right'); axes[0].grid(alpha=0.3)

# PR
base_rate = float(y_test.mean())
pr_aucs = {}
for name, p in [('tuned', y_proba_tuned),
                ('bagged', y_proba_bagged),
                ('isotonic recalibrated', y_proba_iso)]:
    pr, rc, _ = precision_recall_curve(y_test, p)
    pr_auc = auc(rc, pr)
    pr_aucs[name] = pr_auc
    axes[1].plot(rc, pr, label=f'{name} (AUC={pr_auc:.3f})')
axes[1].axhline(base_rate, color='gray', linestyle=':', label=f'base rate = {base_rate:.3f}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall — chronological test set')
axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)
plt.tight_layout()
save_fig('09_roc_pr_curves', fig)
plt.show()

print('\nROC-AUC by source:')
for name, v in roc_aucs.items():
    print(f'  {name:30s} {v:.4f}')
print('PR-AUC by source:')
for name, v in pr_aucs.items():
    print(f'  {name:30s} {v:.4f}')
print(f'Base rate (positive class fraction in test set): {base_rate:.4f}')


In [ ]:
estimator = LogisticRegression(**best_model.get_params())
train_sizes, train_scores, val_scores = learning_curve(
    estimator, X_train_scaled, y_train,
    cv=ts_cv, scoring='f1', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10),
)
tr_m, tr_s = train_scores.mean(1), train_scores.std(1)
vl_m, vl_s = val_scores.mean(1), val_scores.std(1)

print('\nLearning curve — train and CV F1 by training set size:')
lc_df = pd.DataFrame({
    'train_size': train_sizes,
    'train_F1_mean': tr_m,
    'train_F1_std':  tr_s,
    'cv_F1_mean':    vl_m,
    'cv_F1_std':     vl_s,
})
print(lc_df.to_string(index=False, float_format='%.4f'))

fig = plt.figure(figsize=(9, 5))
plt.plot(train_sizes, tr_m, 'o-', color='royalblue', label='Train F1')
plt.fill_between(train_sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color='royalblue')
plt.plot(train_sizes, vl_m, 'o-', color='seagreen', label='CV F1 (TimeSeriesSplit)')
plt.fill_between(train_sizes, vl_m - vl_s, vl_m + vl_s, alpha=0.15, color='seagreen')
plt.xlabel('Training set size'); plt.ylabel('F1 (rain)')
plt.title('Learning curve — TimeSeriesSplit CV')
plt.legend(); plt.grid(linestyle='--', alpha=0.5)
plt.tight_layout()
save_fig('10_learning_curve', fig)
plt.show()


## Finalization

Close the results file and report the artifacts produced.


In [ ]:
log_section('Finalization')

print(f'Results file: {_LR_RESULT_PATH}')
fig_files = sorted(f for f in os.listdir('figures') if f.lower().endswith('.png'))
print(f'Figures saved ({len(fig_files)}):')
for f in fig_files:
    print(f'  figures/{f}')

# Restore stdout and close log file
sys.stdout = _ORIG_STDOUT
_LR_RESULT_FILE.close()
print(f'\nDone. See {_LR_RESULT_PATH} and figures/ for outputs.')
